In [1]:
from src.constants import DATA_PATH
from pathlib import Path

In [2]:
original_folder = Path('/scratch4/odietrich/git/xBD_Sentinel/data/raw/xbd_dataset')
new_folder = DATA_PATH / 'xbd_s12' / 'xbd'
num_workers = 4
overwrite = False

In [7]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Tuple

from osgeo import gdal
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_bounds
from tqdm import tqdm

from src.data.metadata import load_metadata
from src.utils.time import timeit
from src.utils.geometry import reproject_geo
from src.constants import XBD_S12_PATH
from src.data.create_aligned_vrt import create_aligned_vrt_file

df_meta = load_metadata()

row = df_meta.itertuples().__next__()

uid = row.Index
# Calculate corrected spatial info
true_bounds = reproject_geo(row.geometry, "EPSG:4326", row.best_utm).bounds
dst_crs = row.best_utm

# Determine tier folder mapping (xBD specific logic)
tier = "tier1" if row.xbd_tier == "train" else row.xbd_tier

tasks = []
for period in ["pre", "post"]:
    src_path = original_folder / tier / "images" / f"{uid}_{period}_disaster.tif"
    dst_path = new_folder / f"{uid}_{period}_disaster.vrt"  # Saved as .vrt
    print(src_path, dst_path)
    print(src_path.exists())
    print(dst_path.exists())

    create_aligned_vrt_file(src_path, dst_path, true_bounds, dst_crs)

/scratch4/odietrich/git/xBD_Sentinel/data/raw/xbd_dataset/tier1/images/guatemala-volcano_00000000_pre_disaster.tif /scratch/odietrich/git/xbd-s12/data/xbd_s12/xbd/guatemala-volcano_00000000_pre_disaster.vrt
True
False
/scratch4/odietrich/git/xBD_Sentinel/data/raw/xbd_dataset/tier1/images/guatemala-volcano_00000000_post_disaster.tif /scratch/odietrich/git/xbd-s12/data/xbd_s12/xbd/guatemala-volcano_00000000_post_disaster.vrt
True
False


/scratch/odietrich/git/xbd-s12/xbds12-env/lib/python3.12/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [10]:
from src.utils.time import timeit
from src.constants import XBD_S12_PATH
from pathlib import Path
from src.data.metadata import load_metadata
import json
import numpy as np
from shapely.wkt import loads as wkt_loads
import rioxarray as rxr
import cv2
from src.data.create_masks import mask_for_polygon

DAMAGE_DICT = {
    "no-damage": 1,
    "minor-damage": 2,
    "major-damage": 3,
    "destroyed": 4,
    "un-classified": 5,  # eg under clouds
}

def create_raster_mask(json_path: Path, out_fp: Path, nodata_value: int = 6):
    # adapted from https://github.com/PaulBorneP/Xview2_Strong_Baseline/blob/master/legacy/create_masks.py


    # Load the json file and transform to a 1024x1024 mask
    data = json.load(open(json_path))
    mask = np.zeros((1024, 1024), dtype="uint8")
    for feat in data["features"]["xy"]:
        poly = wkt_loads(feat["wkt"])
        subtype = feat["properties"]["subtype"]
        _mask = mask_for_polygon(poly)
        mask[_mask > 0] = DAMAGE_DICT[subtype]

    # Add nodata based on images (eg if the original image is cut)
    uid = out_fp.stem.split("_mask")[0]
    fp_pre = XBD_S12_PATH / "xbd" / f"{uid}_pre_disaster.tif"
    fp_post = XBD_S12_PATH / "xbd" / f"{uid}_post_disaster.tif"
    img_pre = rxr.open_rasterio(fp_pre)
    img_post = rxr.open_rasterio(fp_post)

    # Find nodata pixels (are there any valid pixels that are fully black??)
    mask_pre = (img_pre == 0).all(dim="band").values
    mask_post = (img_post == 0).all(dim="band").values
    mask_nodata = (mask_pre + mask_post).astype(int)  # Final mask for nodata values

    # Use one of the images to get the geotransform and crs
    raster = img_post[0]
    raster.rio.set_nodata(0)
    raster.values = np.where(mask_nodata, nodata_value, mask)

    # Save mask
    out_fp.parent.mkdir(parents=True, exist_ok=True)
    raster.rio.to_raster(out_fp, compress="zstd")

In [11]:
uid = 'guatemala-volcano_00000006'

folder = XBD_S12_PATH / 'masks'
folder.mkdir(exist_ok=True, parents=True)
df_meta = load_metadata()
df_meta.head(10)

,disaster,disaster_type,peril,xbd_tier,event_split,best_utm,N_intact,N_minor,N_major,N_destroyed,...,s2_cs_post,s1_date_pre,s1_date_post,s1_orbit_pre,s1_orbit_post,s1_direction_pre,s1_direction_post,s1_ids_pre,s1_ids_post,geometry
xbd_uid,,,,,,,,,,,,,,,,,,,,,
guatemala-volcano_00000000,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,10,0,0,0,...,0.832290,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.8132 14.39158, -90.81325 14.3870..."
guatemala-volcano_00000001,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,0,4,0,0,...,0.858505,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.81316 14.39612, -90.8132 14.3915..."
guatemala-volcano_00000002,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,0,0,0,1,...,0.884884,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83303 14.42975, -90.83308 14.425..."
guatemala-volcano_00000003,guatemala-volcano,volcano,volcano,test,test,EPSG:32615,0,2,0,1,...,0.893014,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83294 14.43882, -90.83299 14.434..."
guatemala-volcano_00000004,guatemala-volcano,volcano,volcano,hold,test,EPSG:32615,2,6,8,4,...,0.888752,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.82838 14.4297, -90.82842 14.4251..."
guatemala-volcano_00000005,guatemala-volcano,volcano,volcano,test,test,EPSG:32615,0,0,0,0,...,0.889637,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.83751 14.44794, -90.83756 14.443..."
guatemala-volcano_00000006,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,97,0,0,0,...,0.860822,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.80411 14.3688, -90.80416 14.3642..."
guatemala-volcano_00000007,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,9,0,0,0,...,0.850537,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.82363 14.43873, -90.82368 14.434..."
guatemala-volcano_00000008,guatemala-volcano,volcano,volcano,train,test,EPSG:32615,0,0,0,0,...,0.844595,2018-02-06,2018-06-24,26,26,DESCENDING,DESCENDING,S1B_IW_GRDH_1SDV_20180206T115329_20180206T1153...,S1A_IW_GRDH_1SDV_20180624T115419_20180624T1154...,"POLYGON ((-90.81871 14.46591, -90.81876 14.461..."


In [12]:
original_folder = Path('/scratch4/odietrich/git/xBD_Sentinel/data/raw/xbd_dataset')
overwrite = False
for row in df_meta.itertuples():
    uid = row.Index
    if uid != 'guatemala-volcano_00000006':
        continue
    out_fp = folder / f"{uid}_mask.tif"

    tier = 'tier1' if row.xbd_tier == 'train' else row.xbd_tier
    json_fp = original_folder / tier / "labels" / f"{uid}_post_disaster.json" # always post
    create_raster_mask(json_fp, out_fp, nodata_value=6)
    break